In [14]:
import json

import torch
from safetensors.torch import load_file, safe_open, save_file

In [4]:
input_file = "hoverfast_crosstissue_best_model.pth"
output_file = "hoverfast_crosstissue_best_model.safetensors"


In [5]:
%%timeit
# 1. Load the existing pickle weights
state_dict = torch.load("hoverfast_crosstissue_best_model.pth", map_location="cpu", weights_only=False)

177 ms ± 6.46 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit
# 1. Load the existing pickle weights
state_dict = torch.load("hoverfast_crosstissue_best_model.pth", map_location="cuda", weights_only=False)

287 ms ± 6.39 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
import torch

checkpoint = torch.load(input_file, map_location="cpu", weights_only=False)

config_keys = [
    "n_classes",
    "in_channels",
    "padding",
    "depth",
    "wf",
    "up_mode",
    "batch_norm",
    "conv_block",
]

config = {key: checkpoint[key] for key in config_keys if key in checkpoint}

# safetensors metadata values MUST be string key-value pairs
metadata = {"config": json.dumps(config)}

# 3. Extract and sanitize state dict tensors
state_dict = checkpoint["model_dict"]
state_dict = {
    key: value.contiguous()
    for key, value in state_dict.items()
    if isinstance(value, torch.Tensor)
}

# 4. Save state dict AND metadata into safetensors format
save_file(state_dict, output_file, metadata=metadata)

In [10]:
state_dict = load_file("hoverfast_crosstissue_best_model.safetensors")

In [29]:
%%timeit
state_dict = load_file("hoverfast_crosstissue_best_model.safetensors",device="cpu")

7.85 ms ± 121 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [30]:
%%timeit
state_dict = load_file("hoverfast_crosstissue_best_model.safetensors",device="cuda")

30.4 ms ± 1.32 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [17]:
%%timeit
model_path = "hoverfast_crosstissue_best_model.safetensors"

with safe_open(model_path, framework="pt", device="cpu") as f:
    # Read the metadata dictionary
    metadata = f.metadata()
    config = json.loads(metadata["config"])

    # Read state dict weights manually if needed
    state_dict = {key: f.get_tensor(key) for key in f}

# print("Extracted Config:", config)
# print("Keys in State Dict:", len(state_dict))

8.17 ms ± 163 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
